# Hand-E gripper playground

Step by step, with the robot connected:

1. Connect to the robot.
2. Open and close the gripper.
3. Look at what the robot sends back, and check the sim ↔ native mapping.

Everything is in this notebook — no imports from other files. The UR3e must be powered,
in **Remote Control**, with the Robotiq URCapX running and the Hand-E scanned.

**Safety:** the gripper will physically open and close. Keep fingers clear.

## Step 1 · Connect to the robot

The Hand-E talks to the Robotiq URCapX XML-RPC server at `http://<host>:49999/`
(slaveId 9). Native units are **percent** (0 = open, 100 = closed). The socket has a
3 s timeout so a bad connection fails fast instead of hanging the kernel.

In [3]:
import socket
import time
import xmlrpc.client

HOST = "192.168.1.4"   # UR3e on PolyScope X
PORT = 49999
SLAVE_ID = 9
SPEED_PCT = 100
FORCE_PCT = 50

# sim finger position (m, per finger): 0.0 = OPEN, 0.025 = CLOSED
# native percent:                      0   = OPEN, 100   = CLOSED  (flip if the test below disagrees)
SIM_OPEN, SIM_CLOSED = 0.0, 0.025
PCT_OPEN, PCT_CLOSED = 0.0, 100.0


def sim_to_native(sim_m):
    frac = (min(max(sim_m, SIM_OPEN), SIM_CLOSED) - SIM_OPEN) / (SIM_CLOSED - SIM_OPEN)
    return PCT_OPEN + frac * (PCT_CLOSED - PCT_OPEN)


def native_to_sim(pct):
    frac = (min(max(pct, 0.0), 100.0) - PCT_OPEN) / (PCT_CLOSED - PCT_OPEN)
    return SIM_OPEN + frac * (SIM_CLOSED - SIM_OPEN)


def read_state():
    pos = float(g.getCurrentPosition(SLAVE_ID, 0, 0, 0, 0, 0))
    obj = int(g.getObjectDetectionFlag(SLAVE_ID))
    return {
        "pos_pct": pos,
        "obj_flag": obj,
        "grasped": obj in (1, 2),
        "sim_finger": native_to_sim(pos),
        "fault": int(g.getFault(SLAVE_ID)),
        "activated": bool(g.isGripperActivated(SLAVE_ID)),
        "connected": bool(g.isGripperConnected(SLAVE_ID)),
    }


socket.setdefaulttimeout(3.0)   # fail fast instead of hanging the kernel
g = xmlrpc.client.ServerProxy(f"http://{HOST}:{PORT}/")
g.activateIfRequired([SLAVE_ID])
g.setSpeed([SLAVE_ID], SPEED_PCT)
g.setForce([SLAVE_ID], FORCE_PCT)
print(f"connected={g.isGripperConnected(SLAVE_ID)} activated={g.isGripperActivated(SLAVE_ID)}")
print("state:", read_state())

connected=True activated=True
state: {'pos_pct': 97.6471, 'obj_flag': 3, 'grasped': False, 'sim_finger': 0.024411775, 'fault': 0, 'activated': True, 'connected': True}


## Step 2 · Open and close the gripper

Direction-independent URCapX calls. Watch the hardware: full open, then full close.

In [4]:
g.openGripper(SLAVE_ID)
time.sleep(1.5)
print("OPEN   ->", read_state())

g.closeGripper(SLAVE_ID)
time.sleep(1.5)
print("CLOSED ->", read_state())

OPEN   -> {'pos_pct': 1.17647, 'obj_flag': 3, 'grasped': False, 'sim_finger': 0.0002941175, 'fault': 0, 'activated': True, 'connected': True}
CLOSED -> {'pos_pct': 98.0392, 'obj_flag': 3, 'grasped': False, 'sim_finger': 0.024509799999999998, 'fault': 0, 'activated': True, 'connected': True}


## Step 3 · What the robot sends, and the mapping

Measure the open/close percents the robot actually reports and compare to `PCT_OPEN` /
`PCT_CLOSED` above. If OPEN reports a higher percent than CLOSED, swap those two.

In [5]:
g.openGripper(SLAVE_ID)
time.sleep(1.5)
open_pct = read_state()["pos_pct"]

g.closeGripper(SLAVE_ID)
time.sleep(1.5)
closed_pct = read_state()["pos_pct"]

print(f"measured OPEN   : {open_pct:5.1f} %   (PCT_OPEN   = {PCT_OPEN})")
print(f"measured CLOSED : {closed_pct:5.1f} %   (PCT_CLOSED = {PCT_CLOSED})")
if open_pct > closed_pct:
    print("-> OPEN % > CLOSED %: swap PCT_OPEN / PCT_CLOSED in Step 1 and rerun.")
else:
    print("-> direction matches. No swap needed.")

measured OPEN   :   1.2 %   (PCT_OPEN   = 0.0)
measured CLOSED :  98.0 %   (PCT_CLOSED = 100.0)
-> direction matches. No swap needed.


Drive a few sim values through the mapping and read back. Put a 4 cm cube between the
fingers before the `0.025` (closed) step to see `grasped` go True.

In [6]:
for sim_value in [0.0, 0.0125, 0.025]:
    pct = sim_to_native(sim_value)
    g.move([SLAVE_ID], pct, 0, [0] * 16)
    time.sleep(1.5)
    st = read_state()
    print(f"sim {sim_value:6.4f} m -> cmd {pct:5.1f} %  |  "
          f"pos {st['pos_pct']:5.1f} %  sim_finger {st['sim_finger']:.4f} m  "
          f"obj_flag {st['obj_flag']}  grasped {st['grasped']}")

sim 0.0000 m -> cmd   0.0 %  |  pos   1.2 %  sim_finger 0.0003 m  obj_flag 3  grasped False
sim 0.0125 m -> cmd  50.0 %  |  pos  50.2 %  sim_finger 0.0125 m  obj_flag 3  grasped False
sim 0.0250 m -> cmd 100.0 %  |  pos  98.0 %  sim_finger 0.0245 m  obj_flag 3  grasped False
